# **Atelier Préparation de Données Textuelles**

## **Partie 1 – Exploration du corpus**

In [1]:
import pandas as pd

### **1.1 Chargement des données du corpus**

In [3]:
corpus = pd.read_csv("../data/smart_reviews_raw.csv")
corpus.head()

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST...,positif,5,fr


### **1.2 Combien d'avis contient le dataset**

In [5]:
print(f"le nombre d'avis du carpus: {corpus.shape[0]} avis")

le nombre d'avis du carpus: 1200 avis


### **1.3 Combien de colonnes possède-t-il ?**

In [6]:
print(f"le nombre de colonnes du carpus: {corpus.shape[1]}")

le nombre de colonnes du carpus: 8


### **1.4 Quel est le type de chaque colonne ?**

In [12]:
corpus.dtypes

id_avis        str
date           str
source         str
produit        str
texte          str
sentiment      str
note         int64
langue         str
dtype: object

### **1.5 Vérification des valeurs manquantes**

In [14]:
corpus.isna().sum()

id_avis      0
date         0
source       0
produit      0
texte        5
sentiment    0
note         0
langue       0
dtype: int64

On observe 5 valeurs manquantes dans au niveau de la colonnne texte

### **1.6 Identifier quelques types de texte en affichant par exemple :** 

**`texte normal ;`** 

**`texte vide ;`** 

**`texte contenant une URL ;`** 

**`texte contenant une mention ;`** 

**`texte contenant un hashtag ;`** 

**`texte contenant des emojis ;`** 

**`texte avec beaucoup de ponctuation ;`** 

**`texte en majuscules ;`** 

**`texte avec répétition de caractères`**

**`texte normal ;`** 

In [18]:
corpus[['texte']].head(1)

,texte
0,"Très bonne expérience, simple et efficace."


**`texte vide ;`** 

In [28]:
print(corpus[corpus['texte'].isna()]['texte'])

46     NaN
108    NaN
310    NaN
640    NaN
781    NaN
Name: texte, dtype: str


In [35]:
def est_vide(texte):
    return pd.isna(texte) or str(texte).strip() == ""

In [36]:
# est_vide travaille sur UN texte : pour l'appliquer à toute la colonne, on utilise .apply()
masque_vide = corpus['texte'].apply(est_vide)
print(f"Nombre de textes vides : {masque_vide.sum()}")
corpus[masque_vide][['id_avis', 'texte']]

Nombre de textes vides : 6


,id_avis,texte
46,AV0047,NaN
108,AV0109,NaN
310,AV0311,NaN
640,AV0641,NaN
781,AV0782,NaN
951,AV0952,


**`texte contenant une URL ;`**

In [37]:
corpus[corpus['texte'].str.contains(r'https?://|www\.', regex=True, na=False)][['texte']].head(5)

,texte
16,"Produit parfait, rien à signaler. https://exam..."
30,"Très bonne expérience, simple et efficace. htt..."
31,"Je viens de recevoir le produit, à voir dans l..."
34,"Produit parfait, rien à signaler. https://exam..."
35,"Très bon produit, la qualité est au rendez-vou..."


**`texte contenant une mention ;`**

In [38]:
corpus[corpus['texte'].str.contains(r'@\w+', regex=True, na=False)][['texte']].head(5)

,texte
9,@client Livraison rapide et produit conforme à...
32,@client Service client réactif et commande reç...
40,@client La qualité est correcte sans être exce...
44,"@client Je recommande vivement, tout fonctionn..."
45,@client Excellent rapport qualité-prix.


**`texte contenant un hashtag ;`**

In [39]:
corpus[corpus['texte'].str.contains(r'#\w+', regex=True, na=False)][['texte']].head(5)

,texte
1,Très satisfait de mon achat 👍 #avis
12,La batterie tient vraiment bien et l'écran est...
15,Produit correct pour son prix. #avis
38,"Très bonne expérience, simple et efficace. #avis"
69,La batterie tient vraiment bien et l'écran est...


**`texte contenant des emojis ;`**

In [40]:
corpus[corpus['texte'].str.contains(r'[\U0001F300-\U0001FAFF☀-➿]', regex=True, na=False)][['texte']].head(5)

,texte
1,Très satisfait de mon achat 👍 #avis
5,Très satisfait de mon achat 👍 😊
7,Très satisfait de mon achat 👍
10,Très satisfait de mon achat 👍 !!!
19,"Je viens de recevoir le produit, à voir dans l..."


**`texte avec beaucoup de ponctuation ;`** (au moins 2 signes `!`, `?` ou `.` consécutifs)

In [41]:
corpus[corpus['texte'].str.contains(r'[!?.]{2,}', regex=True, na=False)][['texte']].head(5)

,texte
3,"Produit excellent, je suis très satisfait. !!!"
10,Très satisfait de mon achat 👍 !!!
14,Qualité décevante pour ce prix. !!!
25,"Produit reçu endommagé, je suis mécontent. !!!"
37,"Produit excellent, je suis très satisfait. !!!"


**`texte en majuscules ;`**

In [42]:
corpus[corpus['texte'].str.isupper().fillna(False)][['texte']].head(5)

,texte
4,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST...
6,SERVICE CLIENT RÉACTIF ET COMMANDE REÇUE RAPID...
17,SERVICE CLIENT RÉACTIF ET COMMANDE REÇUE RAPID...
23,"PRODUIT EXCELLENT, JE SUIS TRÈS SATISFAIT."
24,TRÈS SATISFAIT DE MON ACHAT 👍


**`texte avec répétition de caractères`** (un même caractère 3 fois ou plus d'affilée)

In [43]:
corpus[corpus['texte'].str.contains(r'(.)\1{2,}', regex=True, na=False)][['texte']].head(5)

/var/folders/0k/yt8w_r810_ncllc08w_14jv80000gn/T/ipykernel_42809/2359330564.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  corpus[corpus['texte'].str.contains(r'(.)\1{2,}', regex=True, na=False)][['texte']].head(5)


,texte
3,"Produit excellent, je suis très satisfait. !!!"
10,Très satisfait de mon achat 👍 !!!
14,Qualité décevante pour ce prix. !!!
25,"Produit reçu endommagé, je suis mécontent. !!!"
28,"Produit excellent, je suis trèèès satisfait."


### **1.7 Mesure de la longueur des textes en créant :une nouvelle colonne « longueur » pour déterminer : longueur minimale ; longueur maximale ; longueur moyenne ; médiane et quartiles.**

In [67]:
corpus["longeur"] = corpus['texte'].str.strip().str.len()
corpus[['texte', 'longeur']].head()

,texte,longeur
0,"Très bonne expérience, simple et efficace.",47.0
1,Très satisfait de mon achat 👍 #avis,35.0
2,"Produit parfait, rien à signaler.",37.0
3,"Produit excellent, je suis très satisfait. !!!",46.0
4,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST...,55.0


In [70]:
print(f"longeur min: {corpus['longeur'].min()}")

longeur min: 0.0


In [71]:
print(f"longeur max: {corpus['longeur'].max()}")

longeur max: 635.0


In [72]:
print(f"longeur mediane: {corpus['longeur'].median()}")

longeur mediane: 48.0


In [74]:
print(f"longeur moyenne: {corpus['longeur'].mean().round(2)}")

longeur moyenne: 49.16


In [77]:
print(f"le 1 er quartile: {corpus['longeur'].quantile(.25)}")

le 1 er quartile: 37.0


In [78]:
print(f"le 3 eme quartile: {corpus['longeur'].quantile(.75)}")

le 3 eme quartile: 56.0
